# Community — Planet NICFI 2024 basemap

Planet NICFI tropical basemap (one of the public user-contributed `projects/...` assets in the catalog) over a small AOI in central Africa. Falls back to a catalog-only exploration if EE returns a permission error.

## Setup

The notebook reads the GEE service-account credentials from `GEE_SERVICE_ACCOUNT` / `GEE_SERVICE_KEY` environment variables. Both must be set before running this cell.

In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pyramids.dataset import Dataset as PyramidsDataset

from earthlens.gee import GEE, Catalog

OUT_DIR = Path('out') / 'community'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SERVICE_ACCOUNT = os.environ['GEE_SERVICE_ACCOUNT']
SERVICE_KEY = os.environ['GEE_SERVICE_KEY']
print(f'output directory: {OUT_DIR.resolve()}')

2026-05-17 17:27:51 | INFO | pyramids.base.config | Logging is configured.


output directory: C:\gdrive\algorithms\remote-sensing\earthlens\examples\notebooks\gee\out\community


## Inspect the catalog entry

Before downloading anything, look at what the bundled catalog knows about the asset — bands, cadence, license, provider.

In [2]:
cat = Catalog()
ds = cat.get_dataset('projects/planet-nicfi/assets/basemaps/africa')
print(f'title:               {ds.title}')
print(f'ee_type:             {ds.ee_type}')
print(f'spatial_resolution:  {ds.spatial_resolution} m')
print(f'extent.start_date:   {ds.extent.start_date}')
print(f'extent.end_date:     {ds.extent.end_date}')
print(f'default_reducer:     {ds.default_reducer}')
print(f'license:             {ds.license}')
print(f'provider:            {ds.provider}')
print(f'#bands:              {len(ds.bands)}')
print(f'band ids (first 5):  {list(ds.bands)[:5]}')

title:               Planet NICFI Tropical Africa Basemap (~4.77m, monthly, 2020-)
ee_type:             image_collection
spatial_resolution:  4.77 m
extent.start_date:   2020-09-01
extent.end_date:     None
default_reducer:     median
license:             proprietary
provider:            planet
#bands:              5
band ids (first 5):  ['R', 'G', 'B', 'N', 'Q']


## Download

Tiny AOI ([-1.5, -1.4] lat, [29.5, 29.6] lon) at 100.0 m, `raw` cadence — keeps the synchronous download under EE's 32768-px per-axis cap.

In [3]:
try:
    gee = GEE(
        start='2024-01-01',
        end='2024-03-31',
        variables={'projects/planet-nicfi/assets/basemaps/africa': ['R']},
        lat_lim=[-1.5, -1.4],
        lon_lim=[29.5, 29.6],
        temporal_resolution='raw',
        path=str(OUT_DIR),
        service_account=SERVICE_ACCOUNT,
        service_key=SERVICE_KEY,
        scale=100.0,
        reducer='median',
    )
    paths = gee.download(progress_bar=False)
    print(f'wrote {len(paths)} GeoTIFF(s):')
    for p in paths:
        print(f'  {p}  ({p.stat().st_size / 1024:.1f} KB)')
    download_ok = True
except Exception as exc:
    if True:
        print(f'live EE call failed (tolerated for this category): {type(exc).__name__}: {exc}')
        paths = []
        download_ok = False
    else:
        raise

live EE call failed (tolerated for this category): EEException: ImageCollection.load: ImageCollection asset 'projects/planet-nicfi/assets/basemaps/africa' not found (does not exist or caller does not have access).


## Quick preview

Load the first written GeoTIFF through pyramids and render the single band. (`pyramids.dataset.Dataset` is the project's GeoTIFF/NetCDF wrapper.)

In [4]:
if not download_ok or not paths:
    print('Skipping preview — no GeoTIFF was written.')
else:
    pds = PyramidsDataset.read_file(str(paths[0]))
    arr = pds.read_array()
    if arr.ndim == 3:
        arr = arr[0]
    # Mask the dataset's nodata so the colormap doesn't get pinned to it.
    nodata = pds.no_data_value
    if nodata is not None:
        try:
            arr = np.ma.masked_equal(arr, float(nodata[0] if isinstance(nodata, (list, tuple)) else nodata))
        except (TypeError, ValueError):
            pass
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(arr, cmap='viridis')
    ax.set_title(f'{'projects/planet-nicfi/assets/basemaps/africa'} / {'R'}')
    ax.set_xlabel('x (px)')
    ax.set_ylabel('y (px)')
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()
    print(f'value range: [{float(np.nanmin(arr)):.4g}, {float(np.nanmax(arr)):.4g}]')

Skipping preview — no GeoTIFF was written.


## What's on disk

The GeoTIFF (or empty list, if the EE call was tolerated) is left under the per-notebook `out/` directory for you to inspect. That directory is `.gitignore`d — re-running the notebook overwrites it.

In [5]:
for p in sorted(OUT_DIR.iterdir()) if OUT_DIR.exists() else []:
    print(f'{p}  ({p.stat().st_size / 1024:.1f} KB)')